## 1. Create a search index

In this notebook we will create the search indexes. We will start by importing libraries and env variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)


load_dotenv(override=True)

# Service endpoints
SEARCH_ENDPOINT = os.environ["SEARCH_ENDPOINT"]
AOAI_ENDPOINT = os.environ["AOAI_ENDPOINT"]

# Model deployments
EMBEDDING_MODEL = os.environ.get("AOAI_EMBEDDING_MODEL", "text-embedding-3-large")
EMBEDDING_DEPLOYMENT = os.environ.get("AOAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
GPT_MODEL = os.environ.get("AOAI_AGENTIC_GPT_MODEL", "gpt-4o-mini")
GPT_DEPLOYMENT = os.environ.get("AOAI_AGENTIC_GPT_DEPLOYMENT", "gpt-4o-mini")

# Resource names
INDEX_MANUALS = "product-manuals"
INDEX_CATALOG = "product-catalog"
INDEXED_SOURCE_MANUALS = "product-index-source-manuals"
INDEXED_SOURCE_CATALOG = "product-index-source-catalog"
KNOWLEDGE_BASE_NAME = "product-kb"

credential = DefaultAzureCredential()
print("Configuration loaded.")

Models: embedding=text-embedding-3-large, chat=gpt-4.1


We will create two indexes: one for documents and another for products (jsons) since json have an structure.

In [14]:
from azure.core.exceptions import ResourceNotFoundError

index_manuals = SearchIndex(
    name=INDEX_MANUALS,
    fields=[
        # Projection-ready schema: chunk docs with parent linkage.
        # Index projections require the key field to use keyword analyzer.
        SearchField(name="id", type="Edm.String", key=True, searchable=True, filterable=True, analyzer_name="keyword"),
        SearchField(name="parent_id", type="Edm.String", filterable=True),
        SearchField(name="title", type="Edm.String", searchable=True),
        # Text Split ordinal positions are arriving as strings in this projection path.
        SearchField(name="page", type="Edm.String", filterable=True, sortable=True),
        SearchField(name="content", type="Edm.String", searchable=True),
        SearchField(name="blob_url", type="Edm.String", filterable=True),
        SearchField(
            name="content_embedding",
            type="Collection(Edm.Single)",
            stored=False,
            vector_search_dimensions=3072,
            vector_search_profile_name="hnsw_text_3_large",
        ),
    ],
    vector_search=VectorSearch(
        profiles=[
            VectorSearchProfile(
                name="hnsw_text_3_large",
                algorithm_configuration_name="alg",
                vectorizer_name="azure_openai_text_3_large",
            )
        ],
        algorithms=[HnswAlgorithmConfiguration(name="alg")],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="azure_openai_text_3_large",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AOAI_ENDPOINT,
                    deployment_name=EMBEDDING_DEPLOYMENT,
                    model_name=EMBEDDING_MODEL,
                ),
            )
        ],
    ),
    # Semantic config is required for agentic retrieval
    semantic_search=SemanticSearch(
        default_configuration_name="semantic_config",
        configurations=[
            SemanticConfiguration(
                name="semantic_config",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="title"),
                    content_fields=[SemanticField(field_name="content")],
                ),
            )
        ],
    ),
)

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

# Recreate the index to apply immutable schema changes (e.g., page type).
try:
    index_client.delete_index(INDEX_MANUALS)
    print(f"Removed existing index '{INDEX_MANUALS}'")
except ResourceNotFoundError:
    pass

index_client.create_or_update_index(index_manuals)
print(f"✅ Index '{INDEX_MANUALS}' ready")

Removed existing index 'product-manuals'
✅ Index 'product-manuals' ready


Catalog index:

In [9]:
index = SearchIndex(
    name=INDEX_CATALOG,
    fields=[
        SearchField(name="ProductID", type="Edm.String", key=True, filterable=True, sortable=True),
        SearchField(name="ProductName", type="Edm.String", searchable=True, sortable=True),
        SearchField(name="ProductCategory", type="Edm.String", searchable=True, filterable=True, facetable=True),
        SearchField(name="Price", type="Edm.Double", filterable=True, sortable=True, facetable=True),
        SearchField(name="ProductDescription", type="Edm.String", searchable=True),
        SearchField(name="ProductPunchLine", type="Edm.String", searchable=True),
        SearchField(name="ImageURL", type="Edm.String"),
        SearchField(name="blob_url", type="Edm.String", filterable=True),
        SearchField(
            name="ProductVector",
            type="Collection(Edm.Single)",
            stored=False,
            vector_search_dimensions=3072,
            vector_search_profile_name="hnsw_text_3_large",
        ),
    ],
    vector_search=VectorSearch(
        profiles=[
            VectorSearchProfile(
                name="hnsw_text_3_large",
                algorithm_configuration_name="alg",
                vectorizer_name="azure_openai_text_3_large",
            )
        ],
        algorithms=[HnswAlgorithmConfiguration(name="alg")],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="azure_openai_text_3_large",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AOAI_ENDPOINT,
                    deployment_name=EMBEDDING_DEPLOYMENT,
                    model_name=EMBEDDING_MODEL,
                ),
            )
        ],
    ),
    semantic_search=SemanticSearch(
        default_configuration_name="semantic_config",
        configurations=[
            SemanticConfiguration(
                name="semantic_config",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="ProductName"),
                    content_fields=[
                        SemanticField(field_name="ProductDescription"),
                        SemanticField(field_name="ProductPunchLine"),
                        SemanticField(field_name="ProductCategory"),
                    ],
                ),
            )
        ],
    ),
)

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
index_client.create_or_update_index(index)
print(f"✅ Index '{INDEX_CATALOG}' ready for product catalog documents")

✅ Index 'product-catalog' ready for product catalog documents
